# Phase 0 — Reference stack + nondeterminism reproduction

DetKernels Phase 0: confirm GPU, run a reference inference stack (vLLM) on a small
open model, and reproduce the core claim from "Defeating Nondeterminism in LLM
Inference" — that outputs diverge across batch sizes even under greedy decoding.

Run this top-to-bottom on a Colab GPU runtime (Runtime > Change runtime type > GPU).

At the end this saves `report_phase0.json` with:
- a baseline sanity check (repeated runs at fixed batch size 1 should already be
  bitwise identical)
- repeated runs at batch size 32 (should NOT be identical if the claim holds)
- the first token index where a batch-size-1 run diverges from a batch-size-32 run
  of the same prompt

Download that file and share it back so it can feed the Phase 0 gate decision and
the Phase 1 harness design.

In [ ]:
!nvidia-smi

In [ ]:
# Pinned versions matter for vLLM on Colab — unpinned installs frequently break.
# If this cell fails, check https://docs.vllm.ai for the current Colab-compatible pin.
!pip install -q vllm transformers accelerate

**If Colab prompts you to restart the runtime after the install above, do so, then
continue from the cell below (no need to re-run the install cell).**

In [ ]:
import json
import time

from vllm import LLM, SamplingParams

MODEL = "Qwen/Qwen3-1.7B"  # small model to keep iteration fast, per project.md Phase 0
PROMPT = (
    "Explain in one paragraph why floating point addition is not associative, "
    "and why that matters for reproducibility."
)
MAX_TOKENS = 64
N_REPEATS = 20  # bump toward 100+ once this runs cleanly, per Phase 1 harness plan

# Greedy decoding: isolates kernel-level (reduction-order) nondeterminism from
# sampling randomness. Do NOT set torch determinism flags here — the point of
# this notebook is to observe the stock stack's nondeterminism, not suppress it.
sampling_params = SamplingParams(temperature=0.0, top_p=1.0, max_tokens=MAX_TOKENS)

llm = LLM(model=MODEL, dtype="bfloat16", gpu_memory_utilization=0.85)

In [ ]:
def run_batch(prompt: str, batch_size: int):
    """Submit `batch_size` concurrent copies of the same prompt; return the
    generated token ids for the FIRST sequence in the batch. Concurrency is what
    changes vLLM's internal reduction order, not the prompt content."""
    prompts = [prompt] * batch_size
    outputs = llm.generate(prompts, sampling_params, use_tqdm=False)
    return outputs[0].outputs[0].token_ids


def first_divergence(seq_a, seq_b):
    for i, (a, b) in enumerate(zip(seq_a, seq_b)):
        if a != b:
            return i
    if len(seq_a) != len(seq_b):
        return min(len(seq_a), len(seq_b))
    return None  # identical

In [ ]:
# Baseline sanity check: fixed batch size 1, repeated. Should already be
# deterministic — if this ISN'T identical across runs, something other than
# batch-size-dependent reduction order is at play (e.g. nondeterministic CUDA
# kernels regardless of batch, or a sampling bug) and that needs to be run down
# before trusting any batch-size comparison below.
bs1_runs = [run_batch(PROMPT, batch_size=1) for _ in range(N_REPEATS)]
bs1_all_identical = all(r == bs1_runs[0] for r in bs1_runs)
print("batch_size=1 repeated runs all identical:", bs1_all_identical)

In [ ]:
# Repeated runs at batch size 32 — this is the condition expected to reveal
# nondeterminism per the Thinking Machines / SGLang writeups.
bs32_runs = [run_batch(PROMPT, batch_size=32) for _ in range(N_REPEATS)]
bs32_all_identical = all(r == bs32_runs[0] for r in bs32_runs)
print("batch_size=32 repeated runs all identical:", bs32_all_identical)

bs32_divergence_points = []
for i in range(1, len(bs32_runs)):
    d = first_divergence(bs32_runs[0], bs32_runs[i])
    if d is not None:
        bs32_divergence_points.append(d)
print("first-divergence token indices vs run 0:", bs32_divergence_points)

In [ ]:
# Cross comparison: batch_size=1 output vs a batch_size=32 output for the SAME
# prompt. This is the direct batch-size-dependent divergence the project targets.
cross_divergence = first_divergence(bs1_runs[0], bs32_runs[0])
print("first token index where bs=1 output diverges from bs=32 output:", cross_divergence)

In [ ]:
report = {
    "model": MODEL,
    "prompt": PROMPT,
    "max_tokens": MAX_TOKENS,
    "n_repeats": N_REPEATS,
    "timestamp": time.strftime("%Y-%m-%dT%H:%M:%S"),
    "baseline_bs1_all_identical": bs1_all_identical,
    "bs32_all_identical": bs32_all_identical,
    "bs32_first_divergence_token_indices": bs32_divergence_points,
    "bs1_vs_bs32_first_divergence_token_index": cross_divergence,
    "bs1_run0_tokens": list(bs1_runs[0]),
    "bs32_run0_tokens": list(bs32_runs[0]),
}

with open("report_phase0.json", "w") as f:
    json.dump(report, f, indent=2)

print(json.dumps(report, indent=2))

## Next steps

Download `report_phase0.json` (Files pane on the left, or `from google.colab import
files; files.download("report_phase0.json")`) and bring it back to the main repo.

If `bs32_all_identical` is `True` and `bs1_vs_bs32_first_divergence_token_index` is
`None`, the stock stack didn't show the expected nondeterminism with these
settings — try a longer `MAX_TOKENS`, a different prompt, or check the installed
vLLM version against what the original writeups tested against, before concluding
the claim doesn't reproduce.